# Como Usar os Microserviços do NVIDIA DLI

Este notebook mostra exemplos práticos de como interagir com os microserviços provisionados no ambiente.

## ⚠️ Nota Importante
Este notebook **só funciona no ambiente NVIDIA DLI**. Se você estiver no Google Colab ou localmente, as chamadas ao `docker_router` falharão.

## 1️⃣ Imports Necessários

In [ ]:
import requests
import json
from pprint import pprint

: 

## 2️⃣ Conectar ao Docker Router e Obter Ajuda

In [ ]:
# Testar conexão com o docker_router
try:
    response = requests.get("http://docker_router:8070/help", timeout=5)
    print("✅ Docker Router está acessível!")
    print("\nComandos disponíveis:")
    pprint(response.json())
except Exception as e:
    print(f"❌ Erro ao conectar ao docker_router: {e}")
    print("Você pode estar fora do ambiente NVIDIA DLI")

## 3️⃣ Listar Todos os Containers

In [ ]:
try:
    response = requests.get("http://docker_router:8070/containers")
    containers = response.json()
    
    print(f"Total de containers: {len(containers)}\n")
    
    for container in containers:
        status = "🟢" if container.get('status') == 'running' else "🔴"
        print(f"{status} {container.get('name'):<30} | Status: {container.get('status'):<10} | Imagem: {container.get('image')}")
except Exception as e:
    print(f"❌ Erro: {e}")

## 4️⃣ Filtrar Apenas Containers em Execução

In [ ]:
try:
    response = requests.get("http://docker_router:8070/containers")
    containers = response.json()
    
    running_containers = [c for c in containers if c.get('status') == 'running']
    
    print(f"Microserviços em execução ({len(running_containers)}):")
    print("-" * 50)
    for container in running_containers:
        print(f"  • {container.get('name')}")
except Exception as e:
    print(f"❌ Erro: {e}")

## 5️⃣ Verificar Saúde do Frontend

In [ ]:
# Health check do frontend
try:
    response = requests.get("http://frontend:8090", timeout=5)
    if response.status_code == 200:
        print("✅ Frontend está rodando (Porta 8090)")
        print(f"Status Code: {response.status_code}")
    else:
        print(f"⚠️ Frontend retornou status: {response.status_code}")
except Exception as e:
    print(f"❌ Frontend não está acessível: {e}")

## 6️⃣ Verificar Saúde do LLM Client

In [ ]:
# Health check do LLM client
try:
    response = requests.get("http://llm_client:9000", timeout=5)
    print(f"✅ LLM Client está rodando (Porta 9000)")
    print(f"Status Code: {response.status_code}")
except Exception as e:
    print(f"⚠️ LLM Client: {e}")

## 7️⃣ Fazer uma Chamada de API para o LLM Client (Exemplo)

In [ ]:
# Exemplo de como fazer uma requisição ao LLM client
# Nota: Você precisará saber qual endpoint o LLM client expõe

try:
    # Tente acessar endpoints comuns
    endpoints = ['/health', '/status', '/info', '/help']
    
    for endpoint in endpoints:
        try:
            response = requests.get(f"http://llm_client:9000{endpoint}", timeout=2)
            if response.status_code == 200:
                print(f"✅ {endpoint} - disponível")
                print(f"   Resposta: {response.json() if response.text else 'Vazio'}\n")
        except:
            pass
except Exception as e:
    print(f"Erro: {e}")

## 8️⃣ Verificar Logs de um Container

In [ ]:
# Você pode tentar obter logs de um container via docker_router
# Verifique com /help para saber se esse endpoint está disponível

try:
    # Exemplo de como seria a chamada
    response = requests.get("http://docker_router:8070/logs?container=llm_client")
    print(response.json())
except Exception as e:
    print(f"Endpoint de logs não disponível ou erro: {e}")

## 📋 Resumo

### Como Usar Microserviços neste Ambiente:

1. **Docker Router** (Porta 8070): Gateway para interagir com containers
   - `GET /help` - Lista de comandos
   - `GET /containers` - Lista de containers

2. **Jupyter** (Porta 9010-9012): Você está aqui!

3. **Frontend** (Porta 8090): Interface web (Gradio)
   - Acesso: `http://frontend:8090` (do notebook)

4. **LLM Client** (Porta 9000): Servidor para chamadas LLM
   - Acesso: `http://llm_client:9000` (do notebook)

### ⚠️ Importante:
- **Não funciona**: `!docker ps -a` (não há acesso ao Docker dentro do container)
- **Funciona**: `requests.get("http://docker_router:8070/containers")`